# PROC LOANによる学生ローン返済プランの比較

## 概要

学資支援オフィスは、代表的な残高27,500ドルを卒業生コホートがどのように返済すべきかを検討するため、`PROC LOAN`を使って4つの返済形態——連邦固定金利の標準プラン、組成手数料付きの民間借換ローン、上限付き変動金利(ARM)ローン、雇用主提供のバイダウン——を比較する。120か月の返済期間にわたり、4つのオファーは提示された当初金利における毎月の返済額と総支払利息で明確に並ぶ:連邦標準プランが最も利息が高く(**10,021.22ドル**、金利6.53%、返済額**312.68ドル**)、民間借換ローンは金利を下げるものの412.50ドルの手数料が加わり(**9,120.20ドル**、金利5.99%、返済額**305.17ドル**)、そして提示金利が低いARM(**7,099.76ドル**、金利4.75%、返済額**288.33ドル**)と雇用主バイダウン(**6,700.67ドル**、金利4.50%、返済額**285.01ドル**)が最も利息負担が小さい。続く`COMPARE`ブロックは、各プランの累積利息・元金返済額・残高を36か月・60か月・120か月時点で報告し、借り手が借換や完済を検討しやすい時点でそれぞれのローンがどこまで償却されているかを示す。

## データソース

| データセット | 行数 | 説明 | 主な変数 |
|---------|------|-------------|---------------|
| `borrowers` | 40 | `call streaminit(20260531)`と`rand('uniform')`を用いてインラインで生成した合成の卒業生コホートのローンプロファイル。比較のために現実的なローン条件を設定するために使用する。 | `student_id`(1001〜1040)、`balance`(卒業時点の元金;この抽出では11,800〜47,750ドル、平均30,771ドル)、`apr`(年率4.10%〜9.10%、平均6.50%)、`term`(120か月または180か月、平均144か月)、`origination_pct`(手数料1.0%〜2.0%、平均1.50%) |

`PROC LOAN`で分析する代表的な借り手(金額27,500ドル、期間120か月、2026年7月開始)は、このコホートの残高・金利分布のやや下寄りの中央付近に位置する。外部データやネットワークデータは使用しない。このコホートは現実的なローン条件を導くために存在し、一対一の比較は単一の代表的なローンについて実行される。

# PROC LOANによる学生ローン返済プランの比較

学生が卒業する際、学資支援オフィスは複数の返済オファーの中から選択できるよう支援しなければならない。`PROC LOAN`(SAS/ETS)はまさにこの目的のために作られている:固定金利・変動金利(ARM)・バイダウンローンを償却計算し、返済額・総利息・償却の進み具合を横並びで比較する。

このノートブックでは以下を行う:

1. 現実的なローン条件を設定するため、合成の卒業生コホートを生成する。
2. `PROC MEANS`でコホートを要約する。
3. 代表的な27,500ドルの残高について4つの返済プランを構築し、`PROC LOAN`(FIXED / ARM / BUYDOWN + COMPARE)で比較する。
4. 推奨する固定金利プランを単独で再実行し、その単体としての経済性を確認する。

すべてインラインの合成データでローカルに実行される——外部ファイルやネットワークアクセスは一切ない。

## ステップ1 — 合成の卒業生コホートを生成する

40人の借り手をシミュレートする。それぞれについて、卒業時点の元金残高、信用度に緩やかに連動するAPR、標準的な返済期間(10年または15年)、組成手数料率を抽出する。シードにより結果は再現可能である。

In [1]:
データ borrowers;
   呼出 streaminit(20260531);
   長さ plan $ 28;
   繰返 student_id = 1001 から 1040;
      /* 卒業時点の元金残高: 9,500ドル - 48,000ドル */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* 信用度別の年率名目APR: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* 標準的な返済期間: 120か月または180か月 */
      もし rand('uniform') < 0.6 なら term = 120;
      他 term = 180;
      /* 元金に対する組成手数料率: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      出力;
   終了;
実行;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## ステップ2 — コホートのプロファイルを確認する

個々のローンをモデル化する前に、残高・金利・期間の分布を確認する。これにより*代表的な*借り手がどのようなものかが分かる——これが後続の一対一比較の基礎となる。

In [2]:
処理 平均 データ=borrowers n mean MIN MAX maxdec=2;
   変数 balance apr term origination_pct;
   見出 balance='元金残高(ドル)'
         apr='年利率(APR、%)'
         term='返済期間(月)'
         origination_pct='組成手数料率(%)';
実行;

                                                  The MEANS Procedure

 Variable         Label                        N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------------------
 balance          元金残高(ドル)                    40       30771.25    11800.00    47750.00
 apr              年利率(APR、%)                  40           6.50        4.10        9.10
 term             返済期間(月)                     40         144.00      120.00      180.00
 origination_pct  組成手数料率(%)                   40           1.50        1.00        2.00
 --------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## ステップ3 — 代表的な借り手向けに4つの返済プランを比較する

2026年7月開始・期間120か月(10年)の代表的な残高27,500ドルを用いて、現実的な4つのオファーを並べる:

- **連邦直接ローン(標準)** — 金利6.53%のシンプルな固定金利ローン。
- **民間借換ローン(手数料あり)** — 金利は5.99%に下がるが、412.50ドルの初期費用(`INIT=`)がかかる。
- **民間変動金利ローン(ARM)** — 金利4.75%の変動ローンで、1回の調整につき1%・生涯上限5%の`CAPS=`、上限金利9.75%の`MAXRATE=`、年次の`ADJUSTFREQ=`、そして`WORSTCASE`キーワードを伴う。
- **雇用主2-1バイダウン** — 4.50%で始まる補助付きプランで、提示スケジュール上は`BDRATES=`により2年目に5.50%、以降6.50%に段階的に上昇する。

`COMPARE`ステートメントは36か月・60か月・120か月時点でのローン横断ビューを要求し、税率22%の`TAXRATE=`と最低期待収益率4%の`MARR=`を指定する。`OUTSUM=`と`OUTCOMP=`はそれぞれローンごとの要約行と比較行を出力する。以下のリストは、各プラン・各時点について**累積支払利息、累積元金返済額、および残高**を報告し、候補間の償却進捗を明確に示す。

> **金利調整に関する注記。** Jennerの`PROC LOAN`は各プランをその提示された**当初**金利で償却するため、ARMの`CAPS`/`MAXRATE`/`WORSTCASE`設定やバイダウンの`BDRATES`の段階は、リスト上ではローン条件として表示されるものの、返済額の計算には反映**されない**——以下のARMとバイダウンの数値は、それぞれ4.75%と4.50%の当初金利を全期間一定として計算したものである。この2つの合計額は、最悪ケースの上限ではなく最良ケース(当初金利)の費用として扱うこと。なお、ARMの1件ごとの要約(`Loan #3:`ブロック)の総利息・総支払額の行は、下の`COMPARE`テーブルより大きい値を表示することがある——これは現行エンジンのARM利息集計に見られる既知の不整合であり、本ノートブック全体で参照している数値は、一貫性のある`COMPARE`テーブルおよび`Summary of Loan Comparisons`の値である。

In [3]:
処理 loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   /* NOTE: PROC LOAN truncates LABEL= at a fixed 30-byte width; keep
      localized labels short (<=30 bytes) to avoid a multi-byte cut. */
   fixed rate=6.53
         見出='連邦標準ローン';

   fixed rate=5.99 init=412.50
         見出='民間借換(手数料)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       見出='民間変動ARM(最悪)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           見出='雇用主2-1バイダウン';

   比較 at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
実行;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: 連邦標準ローン
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: 民間借換(手数料)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: 民間変動ARM(最悪)
    Loan Type:                   Adjustable Rate
    Amount (Principal):                27500.00
    Interest Rate (annual %):            4.7500
    L


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## ステップ4 — 推奨する固定金利プランを単独で再実行する

返済額の確実性を重視する借り手にとって、連邦固定金利の標準プランが安全な既定選択となる。これを単独で再実行し、4プラン比較で見られたのと同じ主要な数値——月々の返済額**312.68ドル**、総支払額**37,521.22ドル**、総利息**10,021.22ドル**——を、単独のローン要約として確認する。

In [4]:
処理 loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed 見出='連邦標準ローン';
実行;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: 連邦標準ローン
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: 連邦標準ローン
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         21035.90        3752.12        1301.15        2450.97         1858


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## 結果の解釈

4つのプランはいずれも同じ27,500ドルの元金を120か月で完全に償却する(COMPAREテーブルでは、どのプランも120か月目の残高が0.00ドルになる)。したがって判断のポイントは**毎月の返済額**と**提示金利における総利息**である:

| プラン | 金利 | 返済額 | 総利息 | 備考 |
|------|------|---------|----------------|-------|
| 連邦直接ローン(標準) | 6.53% | 312.68ドル | 10,021.22ドル | 手数料なし;借り手保護が最も手厚い |
| 民間借換ローン | 5.99% | 305.17ドル | 9,120.20ドル | 412.50ドルの組成手数料 |
| 民間変動金利ローン(ARM) | 4.75%(当初) | 288.33ドル | 7,099.76ドル | 金利は上昇し得る;数値は当初金利での費用 |
| 雇用主2-1バイダウン | 4.50%(当初) | 285.01ドル | 6,700.67ドル | 雇用継続が前提 |

- **連邦標準**プランは利息面で最も高額(10,021.22ドル)だが、手数料なしで固定・予測可能な312.68ドルの返済額を提供する。
- **民間借換ローン**は金利を5.99%に下げ(利息9,120.20ドルで連邦プランより901ドル少ない)、412.50ドルの組成手数料がかかるため、連邦プランに対する正味の優位性は利息差から手数料を差し引いておよそ488ドルにとどまる——ローンが借換されずに十分長く続く場合にのみ意味を持つ。
- **ARM**と**バイダウン**はここでは利息が最も低く見える(それぞれ7,099.76ドルと6,700.67ドル)が、これは単に**当初**金利が固定オファーより大幅に低いためである。ステップ3で述べた通り、Jennerはこれらの当初金利を一定として保持するため、これらは最良ケースの数値である:実際にはARMが金利上昇したり、バイダウンが雇用主補助を失ったりすれば、数値はより高くなり、返済額は保証されない。

`COMPARE`テーブルは、各プランがどれだけ速く償却されるかを示すことでこの点をさらに明確にする。36か月時点で、連邦プランは利息4,792.27ドルを支払い、元金6,464.10ドルを返済済み(残高21,035.90ドル)である一方、バイダウンは利息3,263.97ドルのみを支払い、元金6,996.24ドルを返済済み(残高20,503.76ドル)である——低金利のプランは利息が少なく、最初の3年間で元金の減りも速い。リスク回避志向の卒業生にとっては、連邦標準プランの金利の確実性が高い利息を正当化することが多く、安定した雇用継続に自信がある借り手にとっては、実際に金利を固定するオプションの中ではバイダウンの補助付き当初金利が最大の節約をもたらす。